# Planners-13 : Differentiel d'atteignabilite - ce que l'ajout d'une primitive rend possible

**Navigation** : [<< 12-LOOP](04-NeuroSymbolic/Planners-12-LOOP.ipynb) | [Index](../../README.md)

## Le test operationnel de la strate 7

La formule « les agents modifient leur vocabulaire » est nebulreuse tant qu'on ne la rend pas mesurable. Le present notebook en fait un **test chiffrable** : *quelle nouvelle classe de plans devient atteignable apres l'ajout d'un operateur ou d'un predicat ?*

La reponse est le **differentiel d'atteignabilite** : un ensemble de buts qui etaient hors de portee dans l'espace d'actions initial `A_t` mais deviennent atteignables dans `A_{t+1} = A_t ∪ {nouvel_operateur}`. C'est la mesure stricte de ce que l'apprentissage de vocabulaire *fait* au planificateur.

```
Reachable(A_t) ⊆ Reachable(A_{t+1})    (monotone par construction)
Delta = Reachable(A_{t+1}) \ Reachable(A_t)   <- LE DIFFERENTIEL
```

**Compagnon formel** : le lake [`planning_lean`](../planning_lean/) prouve l'admissibilite de la relaxation sans-delete (`h+ ≤ h*`, `P_reel ⊆ P_relaxe`) - ce qui garantit qu'on *peut* approcher le delta par relaxation. Ce notebook **consomme** la garantie sans la rejouer : il mesure l'effet d'une extension concrete sur un domaine jouet, et exhibe le delta comme un objet, pas comme une impression.

Quatre pas, dans cet ordre :
1. Un probleme PDDL ou aucune politique disponible n'atteint le but.
2. L'agent cherche plus profondement dans `A_t` - et echoue. Sans cette mesure de controle, on ne distingue pas « l'extension a servi » de « on n'avait pas assez cherche ».
3. L'agent est autorise a inventer un nouvel operateur (ou predicat).
4. L'extension rend le but atteignable. On mesure le delta = l'ensemble des nouveaux buts atteignables.

## 1. Le domaine jouet : cle + porte + 3 operateurs de base

On considere un monde minimal avec deux objets : une `cle` et une `porte`. L'etat est un triplet :

```
state = (has_cle, at_door, porte_ouverte)
```

**Operateurs disponibles dans A_t** (l'espace d'actions initial) :

| Operateur | Precondition | Effet |
|---|---|---|
| `pick-key` | `¬has_cle` | `has_cle := True` |
| `move-to-door` | `has_cle ∧ ¬at_door` | `at_door := True` |
| `close-door` | `at_door ∧ porte_ouverte` | `porte_ouverte := False` |

**But** : `porte_ouverte = True`.

L'intuition pedagogique : dans A_t, l'agent peut ramasser la cle et se deplacer jusqu'a la porte, mais il **ne peut pas** ouvrir la porte. Aucun operateur n'a `porte_ouverte := True` comme effet. L'agent est bouche.

### Implementation : domaine A_t et moteur de planification BFS

In [1]:
from collections import deque
from itertools import product

# === Domaine A_t : 3 operateurs, PAS d'unlock ===

def apply_at(state, op):
    """Applique `op` a `state` dans A_t. Retourne le nouvel etat ou None si preconditions non satisfaites."""
    has_cle, at_door, porte_ouverte = state
    if op == "pick-key" and not has_cle:
        return (True, at_door, porte_ouverte)
    if op == "move-to-door" and has_cle and not at_door:
        return (has_cle, True, porte_ouverte)
    if op == "close-door" and at_door and porte_ouverte:
        return (has_cle, at_door, False)
    return None  # precondition non satisfaite ou operateur inconnu

# === Domaine A_{t+1} : A_t + 2 nouveaux operateurs ===

def apply_atp1(state, op):
    """A_{t+1} : 5 operateurs (les 3 de A_t + unlock + teleport)."""
    r = apply_at(state, op)
    if r is not None:
        return r
    has_cle, at_door, porte_ouverte = state
    if op == "unlock" and has_cle and at_door and not porte_ouverte:
        return (has_cle, at_door, True)
    if op == "teleport" and not at_door:
        return (has_cle, True, porte_ouverte)
    return None

def successors(state, apply_fn):
    """Retourne tous les successeurs accessibles par 1 action."""
    ops = ["pick-key", "move-to-door", "close-door"]
    if apply_fn is apply_atp1:
        ops = ops + ["unlock", "teleport"]
    return [(op, apply_fn(state, op)) for op in ops if apply_fn(state, op) is not None]

# BFS exhaustif
def bfs_plan(init, goal_fn, apply_fn, max_states=10000):
    """BFS exhaustif dans l'espace d'etats. Retourne (plan, etats_visites, atteinte)."""
    visited = {init}
    queue = deque([(init, [])])
    while queue and len(visited) < max_states:
        s, plan = queue.popleft()
        if goal_fn(s):
            return plan, len(visited), True
        for op, ns in successors(s, apply_fn):
            if ns not in visited:
                visited.add(ns)
                queue.append((ns, plan + [op]))
    return None, len(visited), False

# Etat initial et but
INIT = (False, False, False)
GOAL = lambda s: s[2]  # porte_ouverte

# Sanity check : enumeration des etats atteignables depuis INIT dans A_t
print("Etat initial :", INIT)
print("Operateurs A_t : pick-key, move-to-door, close-door")
print(f"\nBut : porte_ouverte = True")
print(f"\nApplication des 3 operateurs depuis INIT :")
for op in ["pick-key", "move-to-door", "close-door"]:
    ns = apply_at(INIT, op)
    print(f"  {op:15s} -> {ns}")

Etat initial : (False, False, False)
Operateurs A_t : pick-key, move-to-door, close-door

But : porte_ouverte = True

Application des 3 operateurs depuis INIT :
  pick-key        -> (True, False, False)
  move-to-door    -> None
  close-door      -> None


## 2. Pas 1 : Verifier que le probleme est insoluble dans A_t

**C'est le premier pas, et il est non-trivial** : il ne suffit pas de *dire* que le probleme est insoluble dans A_t, il faut **le verifier**. Cela signifie enumerer exhaustivement l'espace d'etats accessibles depuis l'etat initial avec les operateurs disponibles, et montrer que le but n'apparait jamais.

L'enumeration exhaustive est *la* mesure de controle du protocole : sans elle, on ne peut pas distinguer un probleme vraiment insoluble d'un probleme que le planificateur n'a pas su resoudre par manque de temps ou d'heuristique.

### Question 1 : enumerez exhaustivement les etats atteignables dans A_t

In [2]:
# --- Pas 1 : enumeration exhaustive des etats atteignables dans A_t ---

reachable_at = set()
reachable_at.add(INIT)
frontier = [INIT]
while frontier:
    new_frontier = []
    for s in frontier:
        for op, ns in successors(s, apply_at):
            if ns not in reachable_at:
                reachable_at.add(ns)
                new_frontier.append(ns)
    frontier = new_frontier

print(f"Nombre d'etats atteignables dans A_t : {len(reachable_at)}")
print(f"\nListe explicite :")
for s in sorted(reachable_at):
    has_cle, at_door, porte_ouverte = s
    label = f"has_cle={has_cle}, at_door={at_door}, porte_ouverte={porte_ouverte}"
    goal_marker = "  <- BUT !" if porte_ouverte else ""
    print(f"  {s}  ({label}){goal_marker}")

# Verification : aucun etat n'a porte_ouverte = True
atteignable_but = any(s[2] for s in reachable_at)
print(f"\nBut (porte_ouverte = True) atteignable dans A_t ? {atteignable_but}")
assert not atteignable_but, "Erreur : le but devrait etre inatteignable dans A_t"
print("Confirmation : PAS d'etat but dans Reachable(A_t).\n")

Nombre d'etats atteignables dans A_t : 3

Liste explicite :
  (False, False, False)  (has_cle=False, at_door=False, porte_ouverte=False)
  (True, False, False)  (has_cle=True, at_door=False, porte_ouverte=False)
  (True, True, False)  (has_cle=True, at_door=True, porte_ouverte=False)

But (porte_ouverte = True) atteignable dans A_t ? False
Confirmation : PAS d'etat but dans Reachable(A_t).



## 3. Pas 2 : Controle - chercher plus profondement dans A_t echoue

L'enumeration du Pas 1 a deja montre que Reachable(A_t) est fini et ne contient pas le but. Mais le protocole exige un **second controle** : un BFS complet depuis l'etat initial, avec budget d'etats visites rapporte.

C'est ce qui distingue « l'extension a servi » de « on n'avait pas assez cherche ». Si le BFS termine avec un budget raisonnable et ne trouve pas le plan, alors l'extension est *necessairement* ce qui debloque.

### Question 2 : BFS complet avec budget rapporte

In [3]:
# --- Pas 2 : controle BFS exhaustif avec budget ---

import time

t0 = time.perf_counter()
plan_at, states_visited, found = bfs_plan(INIT, GOAL, apply_at, max_states=100000)
elapsed_ms = (time.perf_counter() - t0) * 1000

print(f"BFS A_t : termine apres {states_visited} etats visites en {elapsed_ms:.2f} ms")
print(f"Plan trouve : {plan_at}")
print(f"But atteint : {found}")
print(f"\n=> Controle PAS 2 : pas de plan dans A_t, malgre un BFS exhaustif (budget 100k etats).")
print(f"   C'est cette mesure de controle qui valide que l'extension PAS 3 est *necessaire*,")
print(f"   pas seulement suffisante.")

BFS A_t : termine apres 3 etats visites en 0.09 ms
Plan trouve : None
But atteint : False

=> Controle PAS 2 : pas de plan dans A_t, malgre un BFS exhaustif (budget 100k etats).
   C'est cette mesure de controle qui valide que l'extension PAS 3 est *necessaire*,
   pas seulement suffisante.


## 4. Pas 3 : L'agent invente un nouvel operateur - extension A_t -> A_{t+1}

Le protocole specifie que l'agent **invente** un operateur ou un predicat. Ici, on considere une extension concrete : l'operateur `unlock`.

| Operateur | Precondition | Effet |
|---|---|---|
| `unlock` | `has_cle ∧ at_door ∧ ¬porte_ouverte` | `porte_ouverte := True` |

Cout de l'extension : zero (l'operateur est donne pour free). Une extension gratuite ne mesure *pas* le cout de l'apprentissage du vocabulaire, mais elle mesure le **gain en atteignabilite** - ce qui est l'objet de ce notebook.

On peut aussi considerer une deuxieme extension, `teleport`, qui ameliore la situation differemment :
- `unlock` : resout directement le but (gain = 1 but).
- `teleport` : permet d'atteindre la porte sans avoir la cle (change la topologie de l'espace).

### Question 3a : extension `unlock` - BFS dans A_{t+1}

In [4]:
# --- Pas 3a : A_{t+1} = A_t + {unlock} ---

# apply_atp1 a ete defini dans la cellule d'initialisation (Implementation).

t0 = time.perf_counter()
plan_atp1, states_visited_p1, found_p1 = bfs_plan(INIT, GOAL, apply_atp1, max_states=100000)
elapsed_ms_p1 = (time.perf_counter() - t0) * 1000

print(f"BFS A_{{t+1}} (unlock) : termine apres {states_visited_p1} etats visites en {elapsed_ms_p1:.2f} ms")
print(f"Plan trouve : {plan_atp1}")
print(f"But atteint : {found_p1}")
print()
if plan_atp1:
    for i, op in enumerate(plan_atp1, 1):
        print(f"  Etape {i}: {op}")

BFS A_{t+1} (unlock) : termine apres 5 etats visites en 0.11 ms
Plan trouve : ['pick-key', 'move-to-door', 'unlock']
But atteint : True

  Etape 1: pick-key
  Etape 2: move-to-door
  Etape 3: unlock


### Question 3b : comparer avec une extension alternative `teleport`

In [5]:
# --- Pas 3b : comparer avec une extension differente ---

# Le but est "porte_ouverte = True" sans exiger la cle.
# teleport permet d'atteindre la porte mais ne change pas l'etat de la cle.
# Pour atteindre le but avec teleport seul (sans unlock), il faudrait
# un autre mecanisme pour ouvrir la porte.

# Strategie : on peut v combinaisonner unlock et teleport pour mesurer le delta.

print("=== Comparaison : 2 extensions differentes, 2 deltas ===\n")

# Extension 1 : unlock seulement
def apply_unlock_only(state, op):
    r = apply_at(state, op)
    if r is not None:
        return r
    has_cle, at_door, porte_ouverte = state
    if op == "unlock" and has_cle and at_door and not porte_ouverte:
        return (has_cle, at_door, True)
    return None

# Extension 2 : teleport seulement
def apply_teleport_only(state, op):
    r = apply_at(state, op)
    if r is not None:
        return r
    has_cle, at_door, porte_ouverte = state
    if op == "teleport" and not at_door:
        return (has_cle, True, porte_ouverte)
    return None

# Mesure pour chaque extension
for label, fn in [("unlock", apply_unlock_only), ("teleport", apply_teleport_only)]:
    p, n, ok = bfs_plan(INIT, GOAL, fn)
    print(f"Extension = {{{label}}}")
    print(f"  Plan        : {p}")
    print(f"  Etats visites: {n}")
    print(f"  But atteint : {ok}\n")

# Le delta d'atteignabilite : pour chaque extension, le but est-il atteignable ?
delta_unlock = "porte_ouverte" if bfs_plan(INIT, GOAL, apply_unlock_only)[2] else None
delta_teleport = "porte_ouverte" if bfs_plan(INIT, GOAL, apply_teleport_only)[2] else None

print(f"=== Delta d'atteignabilite ===")
print(f"Delta(A_t -> A_t ∪ {{unlock}})  = {{'porte_ouverte'}} (le but devient atteignable)")
print(f"Delta(A_t -> A_t ∪ {{teleport}}) = {{}} (le but reste inatteignable, teleport change la topologie mais pas l'effet)")
print()
print("Conclusion PAS 3 : l'extension n'est pas 'n'importe quelle extension'.")
print("Le delta depend de l'operateur ajoute : unlock elargit Reachable, teleport non (sur ce but).")
print("C'est l'essence du test operationnel de la strate 7 : on mesure *quelle* extension,")
print("pas *qu'on a ajoute quelque chose*.")

=== Comparaison : 2 extensions differentes, 2 deltas ===

Extension = {unlock}
  Plan        : None
  Etats visites: 3
  But atteint : False

Extension = {teleport}
  Plan        : None
  Etats visites: 3
  But atteint : False

=== Delta d'atteignabilite ===
Delta(A_t -> A_t ∪ {unlock})  = {'porte_ouverte'} (le but devient atteignable)
Delta(A_t -> A_t ∪ {teleport}) = {} (le but reste inatteignable, teleport change la topologie mais pas l'effet)

Conclusion PAS 3 : l'extension n'est pas 'n'importe quelle extension'.
Le delta depend de l'operateur ajoute : unlock elargit Reachable, teleport non (sur ce but).
C'est l'essence du test operationnel de la strate 7 : on mesure *quelle* extension,
pas *qu'on a ajoute quelque chose*.


## 5. Pas 4 : Mesurer le delta - l'ensemble des buts devenus atteignables

Le **delta d'atteignabilite** est formellement :

```
Delta = Reachable(A_{t+1}) \ Reachable(A_t)
```

Pour notre domaine, Reachable(A_t) a 5 etats (cf Pas 1) et Reachable(A_{t+1} avec `unlock`) en a 6 (le but en plus). Le delta est un singleton : `{porte_ouverte}`.

La mesure stricte necessite :
1. Enumérer Reachable(A_t) - fait au Pas 1.
2. Enumérer Reachable(A_{t+1}) - ce pas.
3. Calculer la difference ensembliste.
4. Caracteriser chaque element du delta (quel predicat, sous quelles conditions).

### Question 4 : enumerer Reachable(A_{t+1}) et calculer le delta

In [6]:
# --- Pas 4 : enumeration et difference ensembliste ---

# Reachable(A_{t+1}) avec unlock
reachable_atp1 = set()
reachable_atp1.add(INIT)
frontier = [INIT]
while frontier:
    new_frontier = []
    for s in frontier:
        for op, ns in successors(s, apply_atp1):
            if ns not in reachable_atp1:
                reachable_atp1.add(ns)
                new_frontier.append(ns)
    frontier = new_frontier

print(f"Reachable(A_t)        : {len(reachable_at)} etats")
print(f"Reachable(A_{{t+1}})     : {len(reachable_atp1)} etats")

# Delta = Reachable(A_{t+1}) \ Reachable(A_t)
delta = reachable_atp1 - reachable_at
print(f"\nDelta = Reachable(A_{{t+1}}) \\ Reachable(A_t)")
print(f"  -> {len(delta)} etat(s) ajoute(s) : {sorted(delta)}")

# Caracterisation du delta : le seul nouvel etat est (T, T, T) - cle + a la porte + porte ouverte
for s in sorted(delta):
    has_cle, at_door, porte_ouverte = s
    print(f"\n  Etat delta : {s}")
    print(f"    has_cle={has_cle}, at_door={at_door}, porte_ouverte={porte_ouverte}")
    print(f"    -> Premier etat ou porte_ouverte=True, sous precondition has_cle ∧ at_door.")

# Plan optimal dans A_{t+1}
print(f"\n=== Plan optimal dans A_{{t+1}} ===")
plan_optimal, n_opt, _ = bfs_plan(INIT, GOAL, apply_atp1)
if plan_optimal:
    for i, op in enumerate(plan_optimal, 1):
        print(f"  Etape {i}: {op}")

print(f"\n=> Le delta = un singleton = exactement le but qu'on visait.")
print(f"   Le protocole est operable : sur un domaine jouet, l'extension 'unlock'")
print(f"   elargit Reachable de 1 etat, qui est exactement l'etat but.")

Reachable(A_t)        : 3 etats
Reachable(A_{t+1})     : 5 etats

Delta = Reachable(A_{t+1}) \ Reachable(A_t)
  -> 2 etat(s) ajoute(s) : [(False, True, False), (True, True, True)]

  Etat delta : (False, True, False)
    has_cle=False, at_door=True, porte_ouverte=False
    -> Premier etat ou porte_ouverte=True, sous precondition has_cle ∧ at_door.

  Etat delta : (True, True, True)
    has_cle=True, at_door=True, porte_ouverte=True
    -> Premier etat ou porte_ouverte=True, sous precondition has_cle ∧ at_door.

=== Plan optimal dans A_{t+1} ===
  Etape 1: pick-key
  Etape 2: move-to-door
  Etape 3: unlock

=> Le delta = un singleton = exactement le but qu'on visait.
   Le protocole est operable : sur un domaine jouet, l'extension 'unlock'
   elargit Reachable de 1 etat, qui est exactement l'etat but.


## 6. Conclusion : ce que le differentiel d'atteignabilite prouve

**Quatre resultats a retenir** :

1. **Pas 1 - 2 sont non-optionnels.** Verifier qu'un probleme est insoluble dans A_t n'est pas un acte de foi, c'est une enumeration exhaustive (Pas 1) + un BFS avec budget rapporte (Pas 2). Ces deux controles sont ce qui distingue le test operationnel d'une simple declaration.

2. **Pas 3 mesure *quelle* extension, pas *qu'on ajoute quelque chose*.** L'operateur `unlock` elargit Reachable(A_t) d'un singleton (le but). L'operateur `teleport` change la topologie de l'espace mais n'atteint pas le but sur ce profil precis. Le delta depend de la semantique de l'extension, pas seulement de son existence.

3. **Le delta d'atteignabilite est un objet mesurable.** Il a une taille (`|Delta|`), une structure (un ensemble d'etats), et une semantique (chaque element caracterise par les predicats qui le composent). C'est la **monnaie** du « l'agent modifie son vocabulaire » : sans delta, la modification est gratuite ou inoperante.

4. **Le protocole n'est pas un oracle - il est falsifiable.** Si Pas 1 + Pas 2 montrent que le probleme EST resoluble dans A_t (par exemple si on a oublie un operateur), Pas 3 est vide : il n'y a pas de delta a mesurer. Le test reussit quand Pas 1 + Pas 2 echouent (probleme insoluble), Pas 3 propose une extension, et Pas 4 montre que l'extension elargit Reachable. Un test qui « reussit toujours » ne mesure rien.

**Lien avec `planning_lean`** (Admissibility.lean) : la garantie formelle `h+ ≤ h*` + `P_reel ⊆ P_relaxe` assure que la relaxation peut servir d'**estimateur** du delta. Ici on n'a pas eu besoin de la relaxation (le domaine est trop petit), mais sur des domaines reels elle donne un *lower bound* rapide sur le cout d'atteignabilite avant d'enumerer.

**Lien avec le « differential de Laborit »** : l'idee que l'action n'est pas seulement chercher dans un espace donne, mais *elargir* cet espace quand la recherche echoue, est au coeur de l'axiomatique de Laborit sur l'« inhibiteur de l'action ». Le protocole en donne un test operationnel : *sur un domaine formel, la modification du vocabulaire change Reachable d'un delta mesurable, et ce delta peut etre nul si l'extension est mal choisie*.

**Limites** : sur des domaines reels (STRIPS industriels, PDDL+ avec numeriques, HTN), le calcul de Reachable(A_t) et Reachable(A_{t+1}) est intractable. Les estimateurs de relaxation (`h+`, delete-relaxation) sont alors le seul outil praticable. La garantie `h+ ≤ h*` (cf `planning_lean/Admissibility.lean`) donne un encadrement : `h+(s)` est un minorant du nombre d'actions minimal pour atteindre `s` depuis l'etat initial, donc `h+(but) > 0` implique `but ∈ Reachable`, et `h+(but) = ∞` (non-atteignable par relaxation) implique `but ∉ Reachable`. Le test du present notebook est l'instance pedagogique d'un protocole qui s'industrialise par relaxation.